# Validación de 24 cortes en Colab (GPU)

Ejecuta `scripts/run_model.py <modelo> --against naive --folds` sobre la rama indicada y empaqueta los resultados.
Antes de ejecutar: Entorno de ejecución > Cambiar tipo de entorno > GPU (T4). El token de GitHub se pide por teclado y no se guarda en el cuaderno.

In [ ]:
MODEL = "cnn_lstm"  # cnn_lstm | tcn | booster | transformer | timesfm3 (una por ejecucion)
BRANCH = "main"
REPO = "Cesar2926/Proyecto-IA-Orinoco"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os
from getpass import getpass

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]

token = getpass("Token de GitHub (solo lectura): ")
!git clone --branch {BRANCH} https://{token}@github.com/{REPO}.git orinoco
%cd orinoco
# El token no debe quedar guardado en .git/config
!git remote set-url origin https://github.com/{REPO}.git
del token

In [ ]:
!uv sync {"--extra timesfm" if MODEL == "timesfm3" else ""}
!uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

In [ ]:
# Opcional, antes de los pliegues: ajuste con Optuna (20 pruebas, mismos cortes que el CNN-LSTM).
# Solo existe para cnn_lstm y transformer. Copiar los mejores parámetros al YAML
# del modelo, hacer commit y volver a ejecutar el cuaderno con TUNE = False.
TUNE = False
if TUNE:
    !uv run python -c "from orinoco_forecast.data.datasets import TimeSeriesDataset; TimeSeriesDataset().write_parquet()"
    !uv run python scripts/tune_{MODEL}.py --trials 20
    from google.colab import files as _files
    _files.download(f"reports/tuning/{MODEL}_trials.csv")

In [ ]:
# La primera ejecución genera el Parquet del dataset (--write-dataset).
!uv run python scripts/run_model.py {MODEL} --against naive --folds --write-dataset

In [ ]:
import shutil

from google.colab import files

path = shutil.make_archive(f"folds_{MODEL}", "zip", f"data/predictions/{MODEL}/backtest")
files.download(path)

## Paquete de la presentación (redes con GPU)

Pronostica con TCN, CNN-LSTM y Transformer los 52 cortes del test (mismo protocolo que
`reports/test/`) y guarda cada pronóstico. El dashboard lo lee en la pestaña
**Probar predicción** para comparar los seis modelos sin entrenar redes en el portátil.
Copiar el archivo descargado a `reports/live/forecasts.parquet`. No depende de `MODEL`.

In [ ]:
PACK = False
if PACK:
    !uv run python -c "from orinoco_forecast.data.datasets import TimeSeriesDataset; TimeSeriesDataset().write_parquet()"
    !uv run python scripts/make_report.py --eval test --windows 0 --step-days 14 --fit-once --models tcn cnn_lstm transformer --predictions-out reports/live/forecasts.parquet --output reports/live/ranking_redes.csv
    from google.colab import files as _files
    _files.download("reports/live/forecasts.parquet")